# Phase 4B — External Validation: MU-Glioma-Post

> **Purpose**: Run the full inference pipeline (Steps 1-5) on the MU-Glioma-Post external validation dataset.
> - Step 1: SwinUNETR feature extraction → 4617-D hybrid embedding per scan
> - Step 2: Read baseline volumes from embeddings
> - Step 3: TaViT forward pass → 256-D trajectory per patient
> - Step 4: Ratio head → proportional volume change
> - Step 5: Classification → progressive/stable/responder
> - (Phase 5: LLM narrative — deferred)

**Requires**: GPU, SwinUNETR checkpoint, TaViT checkpoint, MU-Glioma-Post data

In [ ]:
import subprocess, sys
pkgs = ["monai[all]", "nibabel", "einops"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p],
                   capture_output=True)
print("✅ Dependencies installed")


In [ ]:
import os, gc, time, json, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from monai.data import Dataset, DataLoader
from monai.networks.nets import SwinUNETR
from monai.transforms import MapTransform

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Dimensions (must match training) ──
PATCH = (128, 128, 128)
C_FEAT = 384
OCT_DIM = C_FEAT * 8   # 3072
REG_DIM = C_FEAT * 3   # 1152
GLOB_DIM = C_FEAT       #  384
VOL_DIM = 9             #    9
TOTAL_DIM = OCT_DIM + REG_DIM + GLOB_DIM + VOL_DIM  # 4617

PROJ_DIM = 256
MAX_SEQ_LEN = 8

OUTPUT_ROOT = Path("/kaggle/working/mu_glioma_inference")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output: {OUTPUT_ROOT}")
print(f"Embedding dim: {TOTAL_DIM}")


In [ ]:
# ═══════════════════════════════════════════════════════
# DATA DISCOVERY — MU-Glioma-Post format
# ═══════════════════════════════════════════════════════
import nibabel as nib
import monai.transforms as T

# Step 0: Create .nii.gz symlinks from .nii_gz uploads
# (Kaggle auto-extracts .gz, so we upload as _gz and symlink back)
SYMLINK_DIR = Path('/kaggle/working/nifti_links')
n_links = 0
for nii_gz in Path('/kaggle/input').rglob('*.nii_gz'):
    real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
    # Preserve the full Patient/Timepoint directory structure
    # nii_gz.parent = .../MU-Glioma-Post/PatientID_XXXX/Timepoint_N/
    # We want: SYMLINK_DIR/PatientID_XXXX/Timepoint_N/filename.nii.gz
    rel_parts = nii_gz.parts
    # Find the PatientID part in the path
    for pi, part in enumerate(rel_parts):
        if part.startswith("PatientID"):
            sub = Path(*rel_parts[pi:-1]) / real_name
            link = SYMLINK_DIR / sub
            link.parent.mkdir(parents=True, exist_ok=True)
            if not link.exists():
                os.symlink(str(nii_gz), str(link))
                n_links += 1
            break
if n_links > 0:
    print(f"  Created {n_links} symlinks (.nii_gz → .nii.gz) in {SYMLINK_DIR}")
else:
    print(f"  No .nii_gz files found — data may already be .nii.gz format")


# Find MU-Glioma-Post data root (check symlinks first, then raw input)
MU_ROOT = None
for search_root in [SYMLINK_DIR, Path("/kaggle/input")]:
    if not search_root.exists():
        continue
    for candidate in search_root.rglob("PatientID_0003"):
        if candidate.is_dir():
            MU_ROOT = candidate.parent
            break
    if MU_ROOT:
        break

if MU_ROOT is None:
    raise RuntimeError("MU-Glioma-Post data not found — attach dataset")
print(f"MU-Glioma-Post root: {MU_ROOT}")

# Discover all patients and their timepoints
patients = {}
for pdir in sorted(MU_ROOT.iterdir()):
    if not pdir.is_dir() or not pdir.name.startswith("PatientID"):
        continue
    pid = pdir.name
    tps = sorted([d.name for d in pdir.iterdir() if d.is_dir() and d.name.startswith("Timepoint")])
    if len(tps) == 0:
        continue
    patients[pid] = tps

print(f"Total patients: {len(patients)}")
multi_tp = {p: t for p, t in patients.items() if len(t) >= 2}
print(f"TaViT-eligible (≥2 timepoints): {len(multi_tp)}")
print(f"Total scans: {sum(len(t) for t in patients.values())}")

# Build data dicts for MONAI loader
all_dicts = []
for pid, tps in patients.items():
    for tp_name in tps:
        tp_dir = MU_ROOT / pid / tp_name
        t1c = tp_dir / f"{pid}_{tp_name}_brain_t1c.nii.gz"
        t1n = tp_dir / f"{pid}_{tp_name}_brain_t1n.nii.gz"
        t2w = tp_dir / f"{pid}_{tp_name}_brain_t2w.nii.gz"
        t2f = tp_dir / f"{pid}_{tp_name}_brain_t2f.nii.gz"
        mask = tp_dir / f"{pid}_{tp_name}_tumorMask.nii.gz"

        if not all(f.exists() for f in [t1c, t1n, t2w, t2f, mask]):
            continue

        # Extract timepoint number for ordering
        tp_num = int(tp_name.replace("Timepoint_", ""))

        all_dicts.append({
            "image": [str(t1n), str(t1c), str(t2w), str(t2f)],
            "label": str(mask),
            "patient_id": pid,
            "timepoint": tp_num,
        })

print(f"Valid scan dicts: {len(all_dicts)}")
print(f"  Sample: {all_dicts[0]['patient_id']} tp={all_dicts[0]['timepoint']}")

# ── Transforms (identical to Phase3_D1) ──
class ConvertToMultiChannelBrats2024(MapTransform):
    """Labels 1=NCR, 2=ED, 3=ET → [WT, TC, ET]"""
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img==1)|(img==2)|(img==3),  # WT
                (img==1)|(img==3),           # TC
                img==3,                      # ET
            ]
            d[key] = (torch.stack(result, 0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, 0).astype(np.float32))
        return d

val_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats2024(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
print("✅ Transforms ready")


In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 1: LOAD SwinUNETR
# ═══════════════════════════════════════════════════════

model = SwinUNETR(
    in_channels=4, out_channels=3,
    feature_size=48, use_checkpoint=False,
).to(device)

# Find checkpoint
ckpt_search = (
    list(Path("/kaggle/input").rglob("swinunetr_best.pth")) +
    list(Path("/kaggle/input").rglob("swinunetr_best_v4.pth")) +
    list(Path("/kaggle/input").rglob("swinunetr_latest.pth")) +
    [f for f in Path("/kaggle/input").rglob("*.pth") if "nnunet" not in f.name.lower()]
)
if not ckpt_search:
    raise FileNotFoundError("No SwinUNETR checkpoint found — attach model dataset")

ckpt_path = ckpt_search[0]
print(f"Loading: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
model_state = {k.replace("module.", ""): v for k, v in state.items()
               if not k.startswith("emb_head")}
model.load_state_dict(model_state, strict=False)
model.eval()
print("✅ SwinUNETR loaded")


In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 1b: EXTRACT 4617-D HYBRID EMBEDDINGS
# ═══════════════════════════════════════════════════════

def get_wt_bbox(lbl_feat, min_size=2):
    wt = lbl_feat[0]
    mask = (wt > 0.01).nonzero(as_tuple=False)
    if len(mask) < 1: return None
    z_min, y_min, x_min = mask.min(dim=0).values.tolist()
    z_max, y_max, x_max = mask.max(dim=0).values.tolist()
    h, w, d = wt.shape
    z0=max(z_min-1,0); z1=min(z_max+2,h)
    y0=max(y_min-1,0); y1=min(y_max+2,w)
    x0=max(x_min-1,0); x1=min(x_max+2,d)
    if z1-z0 < min_size: z1=min(z0+min_size,h)
    if y1-y0 < min_size: y1=min(y0+min_size,w)
    if x1-x0 < min_size: x1=min(x0+min_size,d)
    return (z0,z1,y0,y1,x0,x1)

# Hook encoder bottleneck
_feats = {}; hooks = []
sv = model.swinViT
target_layer = 'layers3' if hasattr(sv, 'layers3') else 'layers2'
layer_list = getattr(sv, target_layer)
tgt = layer_list[0] if hasattr(layer_list, '__getitem__') else layer_list

def _hk(m, inp, out):
    feat = out[-1] if isinstance(out, (list, tuple)) else out
    _feats['feat'] = feat.detach()

hooks.append(tgt.register_forward_hook(_hk))
print(f"  Hook: swinViT.{target_layer}[0]")

ds = Dataset(all_dicts, val_transforms)
loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
total = len(all_dicts)

embs_arr, ids_arr, tps_arr = [], [], []
n_skip = n_empty = 0
t_start = time.time()

with torch.no_grad():
    for idx, batch in enumerate(loader):
        pid = batch['patient_id'][0]
        tp  = int(batch['timepoint'][0])
        try:
            img = batch['image'].to(device)
            lbl = batch['label'].to(device)

            _feats.clear()
            img = F.interpolate(img, list(PATCH), mode='trilinear', align_corners=False)
            lbl = F.interpolate(lbl, list(PATCH), mode='nearest')
            _ = model(img)

            if 'feat' not in _feats:
                n_skip += 1; continue

            feat = _feats['feat']
            C = feat.shape[1]
            h, w, d = feat.shape[2:]

            lbl_feat = F.adaptive_avg_pool3d(lbl, (h, w, d))

            wt_vol = float(lbl[0,0].sum().item())
            tc_vol = float(lbl[0,1].sum().item())
            et_vol = float(lbl[0,2].sum().item())

            bbox = get_wt_bbox(lbl_feat[0], min_size=2)
            if bbox is not None:
                z0,z1,y0,y1,x0,x1 = bbox
                feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]
                oct_p = F.adaptive_avg_pool3d(feat_crop, (2,2,2))
                oct_vec = oct_p[0].reshape(C, 8).T.reshape(-1).cpu()

                lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]
                feat_flat = feat_crop[0].reshape(C, -1)
                region_vecs = []
                for ch in range(3):
                    mask = lbl_crop[0, ch].reshape(-1)
                    vol_soft = float(mask.sum())
                    if vol_soft > 0.01:
                        rvec = (feat_flat * mask.unsqueeze(0)).sum(1) / mask.sum()
                    else:
                        rvec = torch.zeros(C, device=device)
                    region_vecs.append(rvec.cpu())
            else:
                oct_vec = torch.zeros(8 * C)
                region_vecs = [torch.zeros(C) for _ in range(3)]
                n_empty += 1

            glob_avg = F.adaptive_avg_pool3d(feat, (1,1,1))
            glob_vec = glob_avg[0].reshape(C).cpu()

            log_wt = np.log1p(wt_vol); log_tc = np.log1p(tc_vol); log_et = np.log1p(et_vol)
            has_wt_f = 1.0 if wt_vol > 10 else 0.0
            has_tc_f = 1.0 if tc_vol > 10 else 0.0
            has_et_f = 1.0 if et_vol > 10 else 0.0
            tc_wt = tc_vol / (wt_vol + 1e-6)
            et_wt = et_vol / (wt_vol + 1e-6)
            et_tc = et_vol / (tc_vol + 1e-6)
            vol_vec = torch.tensor([log_wt, log_tc, log_et,
                                    has_wt_f, has_tc_f, has_et_f,
                                    tc_wt, et_wt, et_tc], dtype=torch.float32)

            parts = [oct_vec] + region_vecs + [glob_vec, vol_vec]
            emb = torch.cat(parts).numpy().astype(np.float32)

            embs_arr.append(emb); ids_arr.append(pid); tps_arr.append(tp)

            if idx == 0:
                print(f"\n  [FIRST SCAN] {pid} tp={tp}")
                print(f"    emb dim: {len(emb)} (expected {TOTAL_DIM})")
                print(f"    WT={wt_vol:.0f}v  TC={tc_vol:.0f}v  ET={et_vol:.0f}v")

            if (idx+1) % 50 == 0 or idx == 0:
                elapsed = time.time() - t_start
                rate = (idx+1) / max(elapsed, 1e-6)
                eta = (total-idx-1) / max(rate, 1e-6)
                print(f"  [{idx+1:4d}/{total}] {pid:<20} tp={tp} | {rate:.1f}/s  ETA={eta/60:.1f}m")

            del img, lbl, feat
            torch.cuda.empty_cache() if device.type == 'cuda' else None

        except Exception as ex:
            n_skip += 1
            if idx < 10: print(f"  [{idx+1}] ERROR {pid}: {str(ex)[:100]}")

for h in hooks: h.remove()

arr = np.array(embs_arr)
ids = np.array(ids_arr)
tps_out = np.array(tps_arr)

print(f"\n{'='*50}")
print(f"  Extraction complete: {arr.shape}")
print(f"  Patients: {len(set(ids_arr))} | Scans: {len(arr)}")
print(f"  Skipped: {n_skip} | Empty ROI: {n_empty}")
print(f"  Dim: {arr.shape[1]} (expected {TOTAL_DIM})")

# Save embeddings
np.savez_compressed(OUTPUT_ROOT / "mu_glioma_embeddings.npz",
                    embeddings=arr, patient_ids=ids, timepoints=tps_out)
print(f"  ✅ Saved: {OUTPUT_ROOT / 'mu_glioma_embeddings.npz'}")

# Save volumes CSV
vol_rows = []
for i, (p, t, emb_v) in enumerate(zip(ids_arr, tps_arr, embs_arr)):
    v9 = emb_v[-9:]
    wt_v = float(np.expm1(v9[0])); tc_v = float(np.expm1(v9[1])); et_v = float(np.expm1(v9[2]))
    vol_rows.append({
        "patient_id": p, "timepoint": int(t),
        "wt_vol": wt_v, "tc_vol": tc_v, "et_vol": et_v,
        "has_wt": float(v9[3]>0.5), "has_tc": float(v9[4]>0.5), "has_et": float(v9[5]>0.5),
        "tc_wt_ratio": float(v9[6]), "et_wt_ratio": float(v9[7]), "et_tc_ratio": float(v9[8])
    })
vol_df = pd.DataFrame(vol_rows)
vol_df.to_csv(OUTPUT_ROOT / "mu_glioma_volumes.csv", index=False)
print(f"  ✅ Saved: mu_glioma_volumes.csv ({len(vol_df)} rows)")


In [ ]:
# ═══════════════════════════════════════════════════════
# STEPS 2-3: TaViT TRAJECTORY ENCODING
# ═══════════════════════════════════════════════════════

class ContinuousTimePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_days=3650):
        super().__init__()
        self.d_model = d_model
        self.max_days = max_days

    def forward(self, days_since_baseline):
        B, S = days_since_baseline.shape
        pe = torch.zeros(B, S, self.d_model, device=days_since_baseline.device)
        t = days_since_baseline.float().unsqueeze(-1) / self.max_days
        div_term = torch.exp(
            torch.arange(0, self.d_model, 2, device=days_since_baseline.device).float()
            * (-math.log(10000.0) / self.d_model)
        )
        pe[:, :, 0::2] = torch.sin(t * div_term * self.max_days)
        pe[:, :, 1::2] = torch.cos(t * div_term * self.max_days)
        return pe

class TaViT(nn.Module):
    def __init__(self, input_dim, proj_dim=256, n_heads=8, n_layers=4,
                 dropout=0.2, max_seq_len=8):
        super().__init__()
        self.proj_dim = proj_dim
        self.max_seq_len = max_seq_len
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.time_pe = ContinuousTimePositionalEncoding(proj_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, proj_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=proj_dim, nhead=n_heads,
            dim_feedforward=proj_dim * 4, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.trajectory_head = nn.Sequential(
            nn.LayerNorm(proj_dim), nn.Linear(proj_dim, proj_dim),
        )

    def forward(self, emb_seq, days_seq, padding_mask=None):
        B, S, _ = emb_seq.shape
        x = self.input_proj(emb_seq)
        x = x + self.time_pe(days_seq)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        if padding_mask is not None:
            cls_pad = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
            src_key_padding = torch.cat([cls_pad, padding_mask], dim=1)
        else:
            src_key_padding = None
        token_out = self.transformer(x, mask=None, src_key_padding_mask=src_key_padding)
        cls_out = token_out[:, 0, :]
        trajectory = self.trajectory_head(cls_out)
        return trajectory, token_out

class ProgressionHead(nn.Module):
    def __init__(self, proj_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(proj_dim, 128), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(128, 1),
        )
    def forward(self, trajectory):
        return self.head(trajectory).squeeze(-1)

# ── Load TaViT checkpoint ──
tavit_search = list(Path("/kaggle/input").rglob("tavit_model_final.pt")) + \
               list(Path("/kaggle/input").rglob("tavit_model_last.pt"))
if not tavit_search:
    raise FileNotFoundError("No TaViT checkpoint found — attach dataset")

tavit_path = tavit_search[0]
print(f"Loading TaViT: {tavit_path}")
tavit_ckpt = torch.load(tavit_path, map_location=device, weights_only=False)

tavit = TaViT(input_dim=TOTAL_DIM, proj_dim=PROJ_DIM).to(device)
tavit.load_state_dict(tavit_ckpt["model_state_dict"], strict=False)
tavit.eval()

prog_head = ProgressionHead(PROJ_DIM).to(device)
if "prog_head_state_dict" in tavit_ckpt:
    prog_head.load_state_dict(tavit_ckpt["prog_head_state_dict"])
prog_head.eval()
print("✅ TaViT + ProgressionHead loaded")

# ── Build per-patient sequences and run TaViT ──
# Group embeddings by patient
from collections import defaultdict
patient_scans = defaultdict(list)
for i, (pid, tp) in enumerate(zip(ids_arr, tps_arr)):
    patient_scans[pid].append((tp, i))

# Only patients with ≥2 scans
tavit_pids = [pid for pid, scans in patient_scans.items() if len(scans) >= 2]
print(f"\nTaViT-eligible patients: {len(tavit_pids)}")

trajectory_embs = {}
prog_preds = {}
baseline_vols = {}

with torch.no_grad():
    for pid in tavit_pids:
        scans = sorted(patient_scans[pid], key=lambda x: x[0])  # sort by timepoint
        scan_indices = [s[1] for s in scans[:MAX_SEQ_LEN]]
        scan_tps = [s[0] for s in scans[:MAX_SEQ_LEN]]

        emb_seq = torch.tensor(
            np.stack([embs_arr[i] for i in scan_indices]),
            dtype=torch.float32
        ).unsqueeze(0).to(device)  # (1, S, 4617)

        # Synthetic 90-day spacing
        days = torch.tensor(
            [(t - scan_tps[0]) * 90.0 for t in scan_tps],
            dtype=torch.float32
        ).unsqueeze(0).to(device)  # (1, S)

        trajectory, _ = tavit(emb_seq, days)  # (1, 256)
        pred = prog_head(trajectory)

        trajectory_embs[pid] = trajectory.squeeze(0).cpu().numpy()
        prog_preds[pid] = float(pred.item())

        # Baseline volume from first scan's volumetric metadata
        first_emb = embs_arr[scan_indices[0]]
        v_baseline = float(np.expm1(first_emb[4608]))  # log1p(wt_vol) at idx 4608
        baseline_vols[pid] = v_baseline

print(f"  Trajectories extracted: {len(trajectory_embs)}")
print(f"  Baseline volumes read: {len(baseline_vols)}")

# Save trajectory embeddings
traj_npz = {pid: emb for pid, emb in trajectory_embs.items()}
np.savez_compressed(OUTPUT_ROOT / "mu_glioma_trajectories.npz", **traj_npz)
print(f"  ✅ Saved: mu_glioma_trajectories.npz")


In [ ]:
# ═══════════════════════════════════════════════════════
# STEPS 4-5: RATIO HEAD + CLASSIFICATION
# Load the 3 saved artifacts from A2 — no retraining needed
# ═══════════════════════════════════════════════════════
import pickle
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.isotonic import IsotonicRegression as _IR

# ── Load ratio head artifacts (saved by A2 after evaluation) ──
artifact_search = list(Path("/kaggle/input").rglob("ratio_head.pt"))
scaler_search   = list(Path("/kaggle/input").rglob("ratio_head_scaler.pkl"))
iso_search      = list(Path("/kaggle/input").rglob("ratio_head_iso_cal.pkl"))

if not artifact_search:
    raise FileNotFoundError(
        "ratio_head.pt not found.\n"
        "Run Phase4_A2_TaViT_Eval.ipynb first and attach its output dataset."
    )

# Load ratio head (257 params, Linear 256→1)
ratio_head = nn.Linear(256, 1)
ratio_head.load_state_dict(torch.load(artifact_search[0], map_location="cpu"))
ratio_head.eval()
print(f"✅ ratio_head loaded from: {artifact_search[0]}")

with open(scaler_search[0], "rb") as f:
    scaler = pickle.load(f)
print(f"✅ scaler loaded from: {scaler_search[0]}")

with open(iso_search[0], "rb") as f:
    iso_cal = pickle.load(f)
print(f"✅ isotonic calibrator loaded from: {iso_search[0]}")

# ── Apply to every TaViT-eligible MU-Glioma patient ──
results = []
for pid in tavit_pids:
    traj       = trajectory_embs[pid]        # (256,)
    v_base     = baseline_vols[pid]           # mm³ from volumetric metadata idx 4608
    prog_score = prog_preds[pid]              # progression head scalar

    # Ratio head: standardize → linear → calibrate → physics
    traj_s    = torch.tensor(scaler.transform(traj.reshape(1, -1)), dtype=torch.float32)
    with torch.no_grad():
        log_ratio_raw = ratio_head(traj_s).squeeze().item()
    log_ratio_cal = float(iso_cal.predict([[log_ratio_raw]])[0])
    change_ratio  = float(np.exp(log_ratio_cal) - 1.0)
    delta_mm3     = v_base * change_ratio
    v_final_est   = v_base + delta_mm3

    # Classification from progression head
    if prog_score > 0.5:
        response = "progressive"
    elif prog_score < -0.5:
        response = "responder"
    else:
        response = "stable"

    results.append({
        "patient_id":           pid,
        "n_timepoints":         len(patient_scans[pid]),
        "wt_vol_baseline_mm3":  round(v_base, 1),
        "progression_score":    round(prog_score, 4),
        "trajectory_class":     response,
        "predicted_ratio":      round(change_ratio, 4),
        "predicted_delta_mm3":  round(delta_mm3, 1),
        "predicted_v_final_mm3":round(v_final_est, 1),
    })

results_df = pd.DataFrame(results)

print(f"\n{'='*60}")
print(f"  INFERENCE RESULTS: {len(results_df)} patients")
print(f"{'='*60}")
print(f"\n  Class distribution:")
print(f"    {results_df['trajectory_class'].value_counts().to_dict()}")
print(f"\n  Baseline volume (mm³):")
print(f"    mean  = {results_df['wt_vol_baseline_mm3'].mean():.0f}")
print(f"    median= {results_df['wt_vol_baseline_mm3'].median():.0f}")
print(f"    range = [{results_df['wt_vol_baseline_mm3'].min():.0f}, {results_df['wt_vol_baseline_mm3'].max():.0f}]")
print(f"\n  Predicted change ratio:")
print(f"    mean  = {results_df['predicted_ratio'].mean():.3f}")
print(f"    median= {results_df['predicted_ratio'].median():.3f}")
print(f"    range = [{results_df['predicted_ratio'].min():.3f}, {results_df['predicted_ratio'].max():.3f}]")

# Save
results_df.to_csv(OUTPUT_ROOT / "mu_glioma_results.csv", index=False)
with open(OUTPUT_ROOT / "mu_glioma_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n  ✅ Saved: mu_glioma_results.csv  +  .json")
print(f"\n  Sample results:")
print(results_df.head(10).to_string(index=False))


In [ ]:
# ═══════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"  MU-GLIOMA-POST INFERENCE — COMPLETE")
print(f"{'='*60}")
print(f"  Scans processed:       {len(embs_arr)}")
print(f"  Patients:              {len(set(ids_arr))}")
print(f"  TaViT trajectories:    {len(trajectory_embs)}")
print(f"  Classifications:       {len(results)}")
print(f"")
print(f"  Outputs:")
print(f"    {OUTPUT_ROOT / 'mu_glioma_embeddings.npz'}     — 4617-D per scan")
print(f"    {OUTPUT_ROOT / 'mu_glioma_volumes.csv'}         — WT/TC/ET volumes")
print(f"    {OUTPUT_ROOT / 'mu_glioma_trajectories.npz'}    — 256-D per patient") 
print(f"    {OUTPUT_ROOT / 'mu_glioma_results.csv'}         — structured clinical output")
print(f"    {OUTPUT_ROOT / 'mu_glioma_results.json'}        — JSON for Phase 5 LLM")
print(f"")
print(f"  ── Next: Phase 5 (LLM Narrative Generation) ──")
print(f"  Load mu_glioma_results.json → feed structured fields to LLM → generate narrative")
